# 🚀 MicroForge: Train a Sub-15 MB Model That Beats Cactus Needle

> **Goal:** Train a 12-layer BitNet b1.58 edge foundation model on the official **DroidCall** mobile intent dataset on a free GPU.
> **Target Output:** Packed binary `microforge.bin` (< 12 MB) that runs on edge CPUs with 0% syntax errors and beats Needle's 78.4% benchmark.

### ⚡ Getting Started:
1. Make sure GPU is selected: **Runtime → Change runtime type → T4 GPU**.
2. Click the **Files (folder icon)** on the left sidebar, and drag & drop `microforge_colab.zip` into it.
3. Click **Runtime → Run all** (or press `Ctrl + F9`).

In [ ]:
# 1. Verify GPU Hardware
!nvidia-smi

In [ ]:
# 2. Install dependencies
!pip install -q torch tokenizers pyyaml

In [ ]:
# 3. Unpack MicroForge Code
import os, zipfile
from google.colab import files

if not os.path.exists('microforge_colab.zip'):
    print('Upload microforge_colab.zip now:')
    uploaded = files.upload()

with zipfile.ZipFile('microforge_colab.zip', 'r') as z:
    z.extractall('.')
print('[+] Extracted microforge_colab.zip successfully!')
print('Ready directories:', os.listdir('.'))

In [ ]:
# 4. Download DroidCall Dataset directly in Colab
!mkdir -p data/droidcall/raw tools_schemas models checkpoints
!wget -q -O data/droidcall/raw/DroidCall_train.jsonl https://huggingface.co/datasets/mllmTeam/DroidCall/resolve/main/DroidCall_train.jsonl
!wget -q -O data/droidcall/raw/DroidCall_test.jsonl https://huggingface.co/datasets/mllmTeam/DroidCall/resolve/main/DroidCall_test.jsonl
print('[+] Datasets downloaded successfully from Hugging Face!')

In [ ]:
# 5. Parse DroidCall into MicroForge Wire Format
!python scripts/download_datasets.py
print('[+] Processed DroidCall into train/val/test splits!')

In [ ]:
# 6. Train the 4,096-token Domain BPE Tokenizer on DroidCall
import json
from pathlib import Path
from microforge.tokenizer.builder import ToolTokenizer

with open('tools_schemas/droidcall_tools.json', 'r') as f:
    tools = json.load(f)['tools']

corpus = []
with open('data/droidcall/droidcall_train.jsonl', 'r') as f:
    for line in f:
        corpus.append(json.loads(line)['query'])

tok = ToolTokenizer(vocab_size=4096)
tok.build_from_tool_schemas(tools, extra_corpus=corpus)
tok.save('models/tokenizer.json')
print(f'[+] Tokenizer trained and saved! Vocab size: {tok.actual_vocab_size}')

In [ ]:
# 7. Train MicroForge 12-Layer BitNet Model on GPU (25 Epochs)
!python scripts/train.py --config configs/droidcall_gpu.yaml --device cuda

In [ ]:
# 8. Run Benchmark Evaluation vs Needle 3 on DroidCall Test Split
!python scripts/evaluate.py --checkpoint checkpoints/best_model.pt --test-data data/droidcall/droidcall_test.jsonl --schemas tools_schemas/droidcall_tools.json --max-samples 200

In [ ]:
# 9. Export to Packed 1.58-bit Binary (< 12 MB) and Auto-Download to Your PC
!python scripts/export.py --checkpoint checkpoints/best_model.pt --out models/microforge.bin

from google.colab import files
files.download('models/microforge.bin')
print('[+] Download complete! You now have the trained sub-12 MB binary ready to deploy.')